In [1]:
#from torch import nn
import pandas as pd
#import numpy as np
from pathlib import Path
import pandera as pa
from nfl.lib import enums

from nfl.data_management.DataManager import DataManager
from sklearn.cluster import HDBSCAN

In [2]:
data_path = (Path.cwd() / "nfl/data").resolve()

In [3]:
data = DataManager.get_data(path_to_json = data_path)

reading /home/peter/personal/my_machine_learning/nfl_analytics/nfl/data/play_by_play_2015.csv
reading /home/peter/personal/my_machine_learning/nfl_analytics/nfl/data/play_by_play_2005.csv
reading /home/peter/personal/my_machine_learning/nfl_analytics/nfl/data/play_by_play_2016.csv
reading /home/peter/personal/my_machine_learning/nfl_analytics/nfl/data/play_by_play_2006.csv
reading /home/peter/personal/my_machine_learning/nfl_analytics/nfl/data/play_by_play_1999.csv
reading /home/peter/personal/my_machine_learning/nfl_analytics/nfl/data/play_by_play_2023.csv
reading /home/peter/personal/my_machine_learning/nfl_analytics/nfl/data/play_by_play_2008.csv
reading /home/peter/personal/my_machine_learning/nfl_analytics/nfl/data/play_by_play_2014.csv
reading /home/peter/personal/my_machine_learning/nfl_analytics/nfl/data/play_by_play_2017.csv
reading /home/peter/personal/my_machine_learning/nfl_analytics/nfl/data/play_by_play_2009.csv
reading /home/peter/personal/my_machine_learning/nfl_analyti

In [4]:
data = data[data["play_type"] != enums.PlayType.PUNT.value]
pd.set_option('display.max_columns', None)
data.head(5)

,yardline_100,game_seconds_remaining,quarter_end,drive,sp,qtr,down,goal_to_go,ydstogo,ydsnet,play_type,yards_gained,shotgun,no_huddle,qb_dropback,qb_kneel,qb_spike,qb_scramble,pass_length,air_yards,yards_after_catch,run_gap,field_goal_result,extra_point_result,score_differential,score_differential_post,ep,epa,wp,home_wp,wpa,home_wp_post,air_wpa,yac_wpa,comp_air_wpa,comp_yac_wpa,first_down_rush,first_down_pass,first_down_penalty,third_down_converted,third_down_failed,fourth_down_converted,fourth_down_failed,incomplete_pass,touchback,interception,fumble_forced,fumble_not_forced,fumble_out_of_bounds,solo_tackle,safety,penalty,tackled_for_loss,fumble_lost,qb_hit,rush_attempt,pass_attempt,sack,touchdown,pass_touchdown,rush_touchdown,return_touchdown,extra_point_attempt,field_goal_attempt,fumble,complete_pass,assist_tackle,passing_yards,receiving_yards,rushing_yards,tackle_with_assist,fumble_recovery_1_yards,fumble_recovery_2_yards,return_yards,penalty_yards,replay_or_challenge,replay_or_challenge_result,penalty_type,season,cp,cpoe,series,series_success,series_result,order_sequence,play_clock,play_type_nfl,st_play_type,end_yard_line,drive_play_count,drive_first_downs,drive_ended_with_score,drive_quarter_start,drive_quarter_end,drive_yards_penalized,drive_start_transition,drive_end_transition,drive_game_clock_start,drive_game_clock_end,drive_start_yard_line,drive_end_yard_line,result,div_game,roof,surface,temp,wind,aborted_play,success,pass,rush,first_down,special,play,out_of_bounds,xyac_mean_yardage,xyac_median_yardage,xyac_success,xyac_fd,xpass,pass_oe,day_of_season
0,NaN,3600.0,False,NaN,False,1,NaN,False,0,NaN,NaN,NaN,False,False,<NA>,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.239785,0.000000,0.422024,0.577976,0.000000,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,False,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,False,NaN,NaN,2015,NaN,NaN,1,True,First down,1.0,0.0,GAME_START,NaN,100.0,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,100.0,6,False,outdoors,grass,88.0,13.0,False,False,False,False,<NA>,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,43
1,35.0,3600.0,False,1.0,False,1,NaN,False,0,18.0,kickoff,0.0,False,False,False,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.239785,0.000000,0.422024,0.577976,0.000000,0.577976,NaN,NaN,0.000000,0.000000,False,False,False,False,False,False,False,False,True,False,False,False,False,False,0.0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,NaN,NaN,NaN,False,NaN,NaN,0.0,NaN,False,NaN,NaN,2015,NaN,NaN,1,True,First down,36.0,0.0,KICK_OFF,NaN,25.0,6.0,1.0,False,1.0,1.0,0.0,KICKOFF,PUNT,15:00,11:33,20.0,38.0,6,False,outdoors,grass,88.0,13.0,False,False,False,False,False,True,False,False,NaN,NaN,NaN,NaN,NaN,NaN,43
2,80.0,3600.0,False,1.0,False,1,1.0,False,10,18.0,pass,3.0,False,False,True,False,False,False,short,3.0,0.0,NaN,NaN,NaN,0.0,0.0,0.239785,-0.337139,0.422024,0.577976,-0.001425,0.579401,-0.001425,0.000000,-0.001425,0.000000,False,False,False,False,False,False,False,False,False,False,False,False,False,True,0.0,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,3.0,3.0,NaN,False,NaN,NaN,0.0,NaN,False,NaN,NaN,2015,0.765811,23.418927,1,True,First down,51.0,12.0,PASS,NaN,23.0,6.0,1.0,False,1.0,1.0,0.0,KICKOFF,PUNT,15:00,11:33,20.0,38.0,6,False,outdoors,grass,88.0,13.0,False,False,True,False,False,False,True,False,4.699278,3.0,0.678964,0.225919,0.456481,54.351911,43
3,77.0,3573.0,False,1.0,False,1,2.0,False,7,18.0,run,2.0,False,False,False,False,False,False,NaN,NaN,NaN,guard,NaN,NaN,0.0,0.0,-0.097354,-0.262481,0.420599,0.579401,-0.017304,0.596705,NaN,NaN,0.000000,0.000000,False,False,False,False,False,False,False,False,False,False,False,False,False,True,0.0,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,NaN,NaN,2.0,False,NaN,NaN,0.0,NaN,F

In [5]:
data[["yardline_100", "drive_start_yard_line"]][data["yardline_100"] >= 50].head(10)

,yardline_100,drive_start_yard_line
2,80.0,20.0
3,77.0,20.0
4,75.0,20.0
5,65.0,20.0
6,65.0,20.0
7,62.0,20.0
9,86.0,14.0
10,83.0,14.0
11,75.0,14.0
12,70.0,14.0


In [6]:
data.info()

<class 'pandas.DataFrame'>
Index: 1182399 entries, 0 to 1245911
Columns: 122 entries, yardline_100 to day_of_season
dtypes: boolean(52), float64(48), int64(6), object(2), str(14)
memory usage: 757.8+ MB


In [7]:
for c in data.columns:
    print(c)

yardline_100
game_seconds_remaining
quarter_end
drive
sp
qtr
down
goal_to_go
ydstogo
ydsnet
play_type
yards_gained
shotgun
no_huddle
qb_dropback
qb_kneel
qb_spike
qb_scramble
pass_length
air_yards
yards_after_catch
run_gap
field_goal_result
extra_point_result
score_differential
score_differential_post
ep
epa
wp
home_wp
wpa
home_wp_post
air_wpa
yac_wpa
comp_air_wpa
comp_yac_wpa
first_down_rush
first_down_pass
first_down_penalty
third_down_converted
third_down_failed
fourth_down_converted
fourth_down_failed
incomplete_pass
touchback
interception
fumble_forced
fumble_not_forced
fumble_out_of_bounds
solo_tackle
safety
penalty
tackled_for_loss
fumble_lost
qb_hit
rush_attempt
pass_attempt
sack
touchdown
pass_touchdown
rush_touchdown
return_touchdown
extra_point_attempt
field_goal_attempt
fumble
complete_pass
assist_tackle
passing_yards
receiving_yards
rushing_yards
tackle_with_assist
fumble_recovery_1_yards
fumble_recovery_2_yards
return_yards
penalty_yards
replay_or_challenge
replay_or_chal

In [ ]:
bool_cols = data.select_dtypes(include=['bool', 'boolean']).columns
for col in bool_cols:
    data[col] = data[col].astype('Int64')

numeric_df = data.select_dtypes(include=['number']).fillna(0)

# 2. Find top columns correlated with 'epa' (excluding epa itself)
# Change 'top_n' to include more or fewer features
top_n = 10
correlations = numeric_df.corr()['epa'].abs().drop('epa')
top_features = correlations.nlargest(top_n).index

In [ ]:
top_features

In [ ]:
bool_cols = data.select_dtypes(include=['bool', 'boolean']).columns
for col in bool_cols:
    data[col] = data[col].astype('Int64')
X = data.select_dtypes(include=['number']).fillna(0)
X = X[top_features]
labels = HDBSCAN(min_cluster_size=10).fit_predict(X)


In [ ]:
with pd.option_context('display.max_rows', None):
    print(data.iloc[0])

In [ ]:
data["result"].unique()

In [ ]:
data["series_result", "fixed_drive_result"].head()

In [ ]:
all_df["surface_type"].unique()

In [ ]:
cleaner = DataCleaner("schema.json")
try:
    clean_df = cleaner.clean(all_df)
except pa.errors.SchemaErrors as err:
    print(err.failure_cases)

In [ ]:
clean_df.head()

In [ ]:
all_df["weather"].unique()